In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("environment","")


In [0]:
env_var = dbutils.widgets.get("environment")
print(env_var)

In [0]:
allowed_env_var = ["DEV","TEST","PROD"]

if env_var.upper() not in allowed_env_var:
    raise ValueError(f"environment is not in the allowed environment list {allowed_env_var} ")

In [0]:
config_table = f"retailpulse_config.{env_var.lower()}.databricks_configs"

print(config_table)

note_book_name_var = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath()\
    .get().split("/")[-1]

print(note_book_name_var)

In [0]:
config_df = spark.sql(f"""
                           select * from {config_table}
                           where notebook_name = '{note_book_name_var}'
                           """)

display(config_df.limit(2))

In [0]:
config = {
    row["config_key"]:row["config_value"]

    for row in config_df.collect()
}
print(config)

In [0]:
source_path = config["source_path"]
target_path = config["target_path"]
source_format = config["source_format"]
target_format = config["target_format"]
load_type = config["load_type"]
header = config["header"]
schema_enforcement = config["schema_enforcement"]

print(source_path)
print(target_path)
print(source_format)
print(target_format)
print(load_type)
print(header)
print(schema_enforcement)


In [0]:
stores_df = spark.read.format(source_format)\
            .option("header",header)\
            .option("inferschema","false")\
            .load(source_path)

display(stores_df.limit(2))

In [0]:
stores_df.write.format(target_format).mode(load_type).save(target_path)

In [0]:
bronze_stores_df = spark.read.parquet(target_path)
print(bronze_stores_df.count())
print(stores_df.count())